# SEA Full Pipeline: From Raw Video to Aligned Subtitles

This notebook runs the **complete 4-stage SEA pipeline** on a Thai Sign Language video, starting from a raw `.mp4` file and producing aligned `.vtt` subtitles.

```
Stage 0: Pose Estimation       .mp4  →  .pose     (MediaPipe Holistic)
Stage 1: Sign Segmentation     .pose →  .eaf      (pose_to_segments → SIGN tier)
Stage 2: Embedding (optional)  .pose + .vtt → .npy (SignCLIP / SentenceTransformer)
Stage 3: DP Alignment          .eaf + .vtt  →  aligned .vtt + updated .eaf
```

### What makes this different from a Stage-3-only demo?

A Stage-3-only demo reads human-annotated sign segments directly from a pre-labeled `.eaf` file. This notebook starts from scratch with only:

1. **`04.mp4`** — the raw video
2. **`.vtt` subtitles** — extracted from the `.eaf` file's `CC_Aligned` tier (in production, these would come from ASR)

The human-annotated `Gloss Labeling` tier is kept only as **ground truth for evaluation**.

### Pipeline diagram

```
┌──────────────────────────────────────────────────────────────────────────┐
│                          SEA 4-STAGE PIPELINE                              │
│                                                                            │
│  ┌──────────┐     ┌──────────────────┐     ┌──────────────────┐           │
│  │  Input   │     │   STAGE 0        │     │   STAGE 1        │           │
│  │  Video   │───▶│  Pose Estimation │───▶│  Sign            │           │
│  │  .mp4    │     │  (MediaPipe)     │     │  Segmentation    │           │
│  └──────────┘     │  .mp4 → .pose    │     │  .pose → .eaf    │           │
│                   └──────────────────┘     └────────┬─────────┘           │
│                                                    │                      │
│  ┌──────────┐     ┌──────────────────┐             │                      │
│  │Subtitles │     │   STAGE 2        │             │                      │
│  │ .vtt/srt │───▶│  Embedding       │─────────────┤                      │
│  └──────────┘     │  (SignCLIP /     │  .npy       │                      │
│                   │  SentenceTrans.)│             │                      │
│                   └──────────────────┘             ▼                      │
│                                           ┌──────────────────┐           │
│  ┌──────────┐                      │   STAGE 3        │           │
│  │Subtitles │─────────────────────▶│  DP Alignment    │           │
│  │ .vtt/srt │                      │  (align_dp.py)   │           │
│  └──────────┘                      └────────┬─────────┘           │
│                                           │                      │
│                                  ┌─────────┴────────┐            │
│                                  │  OUTPUT           │            │
│                                  │  *_updated.eaf    │            │
│                                  │  aligned .vtt     │            │
│                                  └──────────────────┘            │
└──────────────────────────────────────────────────────────────────────────┘
```

---
## Section 1 — Environment & Path Setup

### Prerequisites

```bash
# Option A: using uv (recommended)
curl -LsSf https://astral.sh/uv/install.sh | sh   # install uv if needed
uv sync                                             # creates .venv/ and installs all deps
source .venv/bin/activate

# Option B: using pip
python3.12 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Key packages:
- **`pose-format`** — `videos_to_poses` CLI (Stage 0)
- **`sign-language-segmentation`** — `pose_to_segments` CLI (Stage 1)
- **`mediapipe`** — pose estimation backend
- **`sentence-transformers`** — text embedding for Stage 2 (`text_embedding` mode)
- **`numba`** — JIT compiler for the DP inner loop (Stage 3)
- **`pympi-ling`** / **`webvtt-py`** — ELAN and VTT I/O

### GPU support

| Stage | GPU used? | Notes |
|-------|-----------|-------|
| 0 — Pose Estimation | **Yes** (CUDA) | MediaPipe uses GPU if available; CPU fallback on macOS |
| 1 — Segmentation | No | Small model, CPU-only, runs in seconds |
| 2 — Embedding | **Yes** (CUDA) | SentenceTransformer / SignCLIP use GPU if available |
| 3 — DP Alignment | No | Numba JIT on CPU (already very fast) |

In [1]:
import sys
import os
import glob
import shutil
import subprocess
import copy
from pathlib import Path

# ── Repository paths ─────────────────────────────────────────────────────────
REPO_ROOT = Path("..").resolve()
SEA_DIR = REPO_ROOT / "SEA"
MISC_DIR = SEA_DIR / "misc"
DATA_DIR = REPO_ROOT / "data" / "example_alignment"
OUTPUT_DIR = DATA_DIR / "pipeline_output"
ASSETS_DIR = REPO_ROOT / "assets"

# Add SEA modules to import path
for p in [str(SEA_DIR), str(MISC_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Create output subdirectories
for subdir in ["pose_input", "poses", "subtitles", "ground_truth",
               "segmentation", "aligned", "aligned_gt_signs"]:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(exist_ok=True)

# ── GPU / device detection ─────────────────────────────────────────────────
import torch
import numba
import pympi
import mediapipe

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"PyTorch {torch.__version__}  |  CUDA: {CUDA_AVAILABLE}  |  Device: {DEVICE}")
if CUDA_AVAILABLE:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Numba {numba.__version__}  |  MediaPipe {mediapipe.__version__}")
print(f"\nREPO_ROOT:  {REPO_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

PyTorch 2.6.0  |  CUDA: False  |  Device: cpu
Numba 0.62.1  |  MediaPipe 0.10.21

REPO_ROOT:  /Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub
OUTPUT_DIR: /Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub/data/example_alignment/pipeline_output


---
## Section 2 — Preprocessing: Extract VTT from EAF

In production, the `.vtt` comes from ASR or a human interpreter. For this demo, we extract it from the `.eaf` file's `CC_Aligned` tier. The `Gloss Labeling` tier is kept as ground truth.

In [2]:
from utils import eaf_tier_to_vtt

# Find the EAF file (Thai filename — use glob)
eaf_files = glob.glob(str(DATA_DIR / "*.eaf"))
assert len(eaf_files) >= 1, f"No .eaf files found in {DATA_DIR}"
EAF_PATH = eaf_files[0]
print(f"EAF file: {Path(EAF_PATH).name}")

# ── Extract CC_Aligned tier → input VTT ────────────────────────────────────────
subtitle_vtt_path = OUTPUT_DIR / "subtitles" / "04.vtt"
gt_vtt_path = OUTPUT_DIR / "ground_truth" / "04.vtt"

input_cues = eaf_tier_to_vtt(EAF_PATH, "CC_Aligned", str(subtitle_vtt_path))
shutil.copy2(subtitle_vtt_path, gt_vtt_path)

print(f"Extracted {len(input_cues)} subtitle cues from CC_Aligned tier")
print(f"Time range: {input_cues[0]['start']:.1f}s – {input_cues[-1]['end']:.1f}s")
print(f"Written to: {subtitle_vtt_path}")
print(f"Copied to:  {gt_vtt_path}  (ground truth for evaluation)")

EAF file: การเปรียบเทียบและเรียงลำดับ (11.07 นาที).eaf
Extracted 119 subtitle cues from CC_Aligned tier
Time range: 33.6s – 646.6s
Written to: /Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub/data/example_alignment/pipeline_output/subtitles/04.vtt
Copied to:  /Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub/data/example_alignment/pipeline_output/ground_truth/04.vtt  (ground truth for evaluation)


/Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub/.venv/lib/python3.12/site-packages/webvtt/structures.py:5: SyntaxWarning: invalid escape sequence '\d'
  TIMESTAMP_PATTERN = re.compile('(\d+)?:?(\d{2}):(\d{2})[.,](\d{3})')


In [3]:
# ── Extract Gloss Labeling tier → ground truth sign segments (in memory) ───
import xml.etree.ElementTree as ET

def elan_tier_to_cues(eaf_path, tier_name):
    """Extract annotations from any EAF tier into SEA cue-dict format."""
    tree = ET.parse(eaf_path)
    root = tree.getroot()
    time_slots = {}
    for ts in root.find("TIME_ORDER").findall("TIME_SLOT"):
        ts_id = ts.get("TIME_SLOT_ID")
        ts_val = ts.get("TIME_VALUE")
        if ts_val is not None:
            time_slots[ts_id] = float(ts_val) / 1000.0
    tier = None
    for t in root.findall("TIER"):
        if t.get("TIER_ID") == tier_name:
            tier = t
            break
    if tier is None:
        raise ValueError(f"Tier '{tier_name}' not found")
    cues = []
    for ann in tier.findall("ANNOTATION"):
        elem = next(iter(ann), None)
        if elem is None:
            continue
        text_node = elem.find("ANNOTATION_VALUE")
        text = text_node.text if text_node is not None and text_node.text else ""
        start = time_slots.get(elem.get("TIME_SLOT_REF1"))
        end = time_slots.get(elem.get("TIME_SLOT_REF2"))
        if start is not None and end is not None:
            cues.append({"start": start, "end": end, "mid": (start + end) / 2, "text": text})
    return cues

gt_sign_segments = elan_tier_to_cues(EAF_PATH, "Gloss Labeling")
print(f"Ground truth sign segments (Gloss Labeling): {len(gt_sign_segments)}")
print(f"Time range: {gt_sign_segments[0]['start']:.1f}s – {gt_sign_segments[-1]['end']:.1f}s")
print(f"\nFirst 5 segments:")
for seg in gt_sign_segments[:5]:
    print(f"  [{seg['start']:7.2f}s – {seg['end']:7.2f}s]  {seg['text']}")

Ground truth sign segments (Gloss Labeling): 852
Time range: 33.6s – 646.6s

First 5 segments:
  [  33.56s –   34.61s]  สวัสดี
  [  35.09s –   35.77s]  (ผายมือ)
  [  35.99s –   36.09s]  เด็ก
  [  36.29s –   36.62s]  เรียน
  [  36.79s –   37.33s]  (ผายมือ)


---
## Section 3 — Stage 0: Video Crop (Right Half) + Pose Estimation

**Input:** `04.mp4` (raw video)  
**Intermediate:** `04_right_half.mp4` (cropped to right half)  
**Output:** `04.pose` (binary file containing per-frame body/hand/face landmarks)  
**Tool:** `videos_to_poses` CLI from the [`pose-format`](https://github.com/sign-language-processing/pose) package

### What happens inside

1. The notebook crops each frame to the **right half** of the video (`x = width/2 ... width`).
2. `MediaPipe Holistic` then processes the cropped video and extracts:
- 33 body landmarks (BlazePose) + 468 face mesh + 21×2 hand landmarks = **543 landmarks per frame**
- Each landmark has (x, y, z, visibility) coordinates

### GPU acceleration

When CUDA is available, `videos_to_poses` uses the GPU for pose estimation, reducing processing time from ~10 min to ~1–2 min. The `--additional-config` flag allows setting MediaPipe options:
- `model_complexity=2` — highest accuracy (default=1)
- `refine_face_landmarks=true` — more precise face mesh
- `smooth_landmarks=false` — disable temporal smoothing (better for segmentation)

### Performance

| Platform | Time (11-min video) |
|----------|--------------------|
| macOS (CPU) | ~5–15 min |
| Linux (GPU) | ~1–2 min |

The cropped video and pose file are cached — re-running skips computation if outputs already exist.

In [4]:
# ── Stage 0: Crop video (right half) + Pose estimation ─────────────────────
import cv2

RAW_VIDEO_FILE = DATA_DIR / "04.mp4"
CROPPED_VIDEO_DIR = OUTPUT_DIR / "cropped_videos"
CROPPED_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
CROPPED_VIDEO_FILE = CROPPED_VIDEO_DIR / "04_right_half.mp4"

POSE_INPUT_DIR = OUTPUT_DIR / "pose_input"
POSE_DIR = OUTPUT_DIR / "poses"
POSE_FILE = POSE_DIR / "04.pose"

def crop_video_to_right_half(src_path: Path, dst_path: Path):
    """Crop input video to the right half, preserving FPS and audio."""
    cap = cv2.VideoCapture(str(src_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {src_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

    right_x = width // 2
    crop_width = width - right_x

    temp_video = dst_path.with_suffix(".video_tmp.mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(temp_video), fourcc, fps, (crop_width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Failed to create temp cropped video: {temp_video}")

    frame_count = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        cropped = frame[:, right_x:width]
        writer.write(cropped)
        frame_count += 1

    cap.release()
    writer.release()

    if frame_count == 0:
        if temp_video.exists():
            temp_video.unlink()
        raise RuntimeError("No frames were written during crop.")

    # Keep original audio by muxing it back with ffmpeg when available.
    ffmpeg_path = shutil.which("ffmpeg")
    if ffmpeg_path:
        cmd = [
            ffmpeg_path, "-y",
            "-i", str(temp_video),
            "-i", str(src_path),
            "-map", "0:v:0",
            "-map", "1:a?",
            "-c:v", "copy",
            "-c:a", "copy",
            "-shortest",
            str(dst_path),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print("Warning: ffmpeg audio mux failed; using cropped video without audio.")
            print(result.stderr.strip()[:500])
            shutil.move(str(temp_video), str(dst_path))
        else:
            temp_video.unlink(missing_ok=True)
    else:
        print("ffmpeg not found; writing cropped video without audio track.")
        shutil.move(str(temp_video), str(dst_path))

if CROPPED_VIDEO_FILE.exists():
    print(f"Cropped video already exists: {CROPPED_VIDEO_FILE}")
else:
    print("Cropping raw video to right half...")
    crop_video_to_right_half(RAW_VIDEO_FILE, CROPPED_VIDEO_FILE)
    print(f"Done. Cropped video: {CROPPED_VIDEO_FILE}")

# Create a symlink so videos_to_poses can process a clean directory
video_link = POSE_INPUT_DIR / "04.mp4"
if video_link.exists() or video_link.is_symlink():
    video_link.unlink()
os.symlink(CROPPED_VIDEO_FILE.resolve(), video_link)

if POSE_FILE.exists():
    print(f"Pose file already exists: {POSE_FILE}")
    print(f"Size: {POSE_FILE.stat().st_size / 1e6:.1f} MB")
    print("Skipping Stage 0. Delete the file to re-run.")
else:
    print("Running Stage 0: Pose estimation from RIGHT-HALF cropped video...")
    if not CUDA_AVAILABLE:
        print("No GPU detected — running on CPU (may take 5–15 minutes).")
    else:
        print(f"GPU detected ({torch.cuda.get_device_name(0)}) — should take 1–2 minutes.")
    print()

    cmd = [
        "videos_to_poses",
        "--format", "mediapipe",
        "--directory", str(POSE_INPUT_DIR),
        # Higher accuracy settings (same as SEA/README.md example)
        '--additional-config=model_complexity=2,smooth_landmarks=false,refine_face_landmarks=true',
    ]
    print(f"Command: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"STDOUT: {result.stdout}")
        print(f"STDERR: {result.stderr}")
        raise RuntimeError("Pose estimation failed. See output above.")

    # videos_to_poses writes .pose next to .mp4 — move it to our poses/ dir
    generated = POSE_INPUT_DIR / "04.pose"
    if generated.exists():
        shutil.move(str(generated), str(POSE_FILE))
    else:
        candidates = list(POSE_INPUT_DIR.glob("*.pose")) + list(CROPPED_VIDEO_DIR.glob("*.pose")) + list(DATA_DIR.glob("*.pose"))
        if candidates:
            shutil.move(str(candidates[0]), str(POSE_FILE))
        elif not POSE_FILE.exists():
            raise FileNotFoundError("Could not find the generated .pose file")

    print(f"\nDone. Pose file: {POSE_FILE}")
    print(f"Size: {POSE_FILE.stat().st_size / 1e6:.1f} MB")

Cropping raw video to right half...
Done. Cropped video: /Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub/data/example_alignment/pipeline_output/cropped_videos/04_right_half.mp4
Running Stage 0: Pose estimation from RIGHT-HALF cropped video...
No GPU detected — running on CPU (may take 5–15 minutes).

Command: videos_to_poses --format mediapipe --directory /Users/dechathonniamsa-ard/Documents/Dechathon_N/NECTEC/Sign_to_sub/data/example_alignment/pipeline_output/pose_input --additional-config=model_complexity=2,smooth_landmarks=false,refine_face_landmarks=true


KeyboardInterrupt: 

In [ ]:
# ── Verify the pose file ──────────────────────────────────────────────────
from pose_format import Pose

with open(POSE_FILE, "rb") as f:
    pose = Pose.read(f.read())

data = pose.body.data
print(f"Pose data shape: {data.shape}")
print(f"  Frames:     {data.shape[0]}")
print(f"  People:     {data.shape[1]}")
print(f"  Landmarks:  {data.shape[2]}")
print(f"  Dimensions: {data.shape[3]}  (x, y, z, visibility)")
print(f"  FPS:        {pose.body.fps}")
print(f"  Duration:   {data.shape[0] / pose.body.fps:.1f}s")

VIDEO_FPS = int(pose.body.fps)
print(f"\nUsing FPS={VIDEO_FPS} for evaluation.")

---
## Section 4 — Stage 1: Sign Segmentation

**Input:** `04.pose`  
**Output:** `04.eaf` with a `SIGN` tier  
**Tool:** `pose_to_segments` CLI from [`sign-language-segmentation`](https://github.com/J22Melody/segmentation/tree/bsl)

### What happens inside

The segmentation model classifies each frame as:
- **SIGN** — actively signing
- **SIGN-B** (begin) — transition into a new sign
- **SIGN-O** (other) — rest / non-signing

Threshold parameters:
- `sign-b-threshold=30` — lower = more sign onsets detected
- `sign-o-threshold=70` — higher = more frames classified as non-sign

Output: ELAN `.eaf` file with a `SIGN` tier — the machine equivalent of `Gloss Labeling`.

In [ ]:
# ── Stage 1: Sign segmentation ──────────────────────────────────────────
SEG_MODEL = "model_E4s-1.pth"
SIGN_B = 30
SIGN_O = 70

model_name = SEG_MODEL.replace("model_", "").replace(".pth", "")
seg_subdir = OUTPUT_DIR / "segmentation" / f"{model_name}_{SIGN_B}_{SIGN_O}"
seg_subdir.mkdir(parents=True, exist_ok=True)
SEG_OUTPUT = seg_subdir / "04.eaf"

if SEG_OUTPUT.exists():
    print(f"Segmentation file already exists: {SEG_OUTPUT}")
    print("Skipping Stage 1. Delete the file to re-run.")
else:
    print("Running Stage 1: Sign segmentation...")
    cmd = [
        "pose_to_segments",
        "--no-pose-link",
        f"--model={SEG_MODEL}",
        f"--pose={POSE_FILE}",
        f"--elan={SEG_OUTPUT}",
        "--sign-b-threshold", str(SIGN_B),
        "--sign-o-threshold", str(SIGN_O),
    ]
    print(f"Command: {' '.join(str(c) for c in cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"STDOUT: {result.stdout}")
        print(f"STDERR: {result.stderr}")
        raise RuntimeError("Sign segmentation failed.")
    print(f"Done. Segmentation file: {SEG_OUTPUT}")

In [ ]:
# ── Inspect auto-generated SIGN tier vs human Gloss Labeling ───────────────
from utils import get_sign_segments_from_eaf

auto_signs = get_sign_segments_from_eaf(str(SEG_OUTPUT))
print(f"Auto-segmented SIGN tier:     {len(auto_signs):4d} segments")
print(f"Human Gloss Labeling (GT):    {len(gt_sign_segments):4d} segments")
print()
if auto_signs:
    print(f"Auto signs range: {auto_signs[0]['start']:.1f}s – {auto_signs[-1]['end']:.1f}s")
print(f"GT signs range:   {gt_sign_segments[0]['start']:.1f}s – {gt_sign_segments[-1]['end']:.1f}s")
print()
print("First 5 auto-segmented signs:")
for seg in auto_signs[:5]:
    print(f"  [{seg['start']:7.2f}s – {seg['end']:7.2f}s]  duration={seg['end']-seg['start']:.2f}s")

---
## Section 5 — Stage 2: Embedding

Embeddings add a **semantic similarity** signal to the DP cost function. Instead of relying purely on temporal proximity, the aligner can match subtitles to sign groups whose *content* is semantically similar.

### Two embedding modes

| Mode | Model | Requirements | Sign segments need text? | Improvement |
|------|-------|-------------|-------------------------|-------------|
| `text_embedding` | SentenceTransformer (`all-MiniLM-L6-v2`) | None (bundled) | **Yes** — only works with human Gloss Labeling | Moderate |
| `sign_clip_embedding` | [SignCLIP](https://aclanthology.org/2024.emnlp-main.518/) | External model download | **No** — embeds pose features directly | **+6% F1@0.50** |

### How it works

1. **Subtitle embeddings**: Each subtitle cue text is encoded into a 384-dim (SentenceTransformer) or 512-dim (SignCLIP) vector.
2. **Sign embeddings**: Each sign segment is encoded — either from its text label (SentenceTransformer) or from its pose features (SignCLIP).
3. **Similarity matrix**: `sim_matrix[i, j] = dot(subtitle_emb[i], sign_emb[j])` — a (M × N) matrix passed to the DP aligner.
4. The DP cost function subtracts `γ × similarity` from the cost, steering assignments toward semantically matching pairs.

### Configuration

Set `EMBEDDING_MODE` below:
- `"none"` — skip embeddings, temporal alignment only
- `"text_embedding"` — use SentenceTransformer (works out of the box, but requires sign segments with text labels — uses human GT signs)
- `"sign_clip_embedding"` — use pre-computed SignCLIP `.npy` files (requires external setup, see below)

### SignCLIP setup (for `sign_clip_embedding` mode)

```bash
# 1. Clone the fairseq fork with SignCLIP
git clone git@github.com:J22Melody/fairseq.git
cd fairseq
conda env update -n sea -f environment_inference.yml
cd examples/MMPT && pip install .

# 2. Embed signs (from pose features)
python scripts_bsl/extract_episode_features.py \
  --video_ids <video_ids.txt> --mode=segmentation \
  --model_name bsl --language_tag "<en> <bfi>" --batch_size=32 \
  --segmentation_dir <segmentation_dir> \
  --save_dir <segmentation_embedding_dir>

# 3. Embed subtitles (from text)
python scripts_bsl/extract_episode_features.py \
  --video_ids <video_ids.txt> --mode=subtitle \
  --model_name bsl --language_tag "<en> <bfi>" --batch_size=1024 \
  --subtitle_dir <subtitle_dir> \
  --save_dir <subtitle_embedding_dir>
```

Place the resulting `.npy` files in `pipeline_output/segmentation_embedding/` and `pipeline_output/subtitle_embedding/`.

In [ ]:
# ============================================================================
# CONFIGURATION: Choose embedding mode
# ============================================================================
# Options:
#   "none"                 — No embeddings, temporal alignment only (fastest)
#   "text_embedding"       — SentenceTransformer (works out of the box, uses
#                            human GT sign segments since auto signs lack text)
#   "sign_clip_embedding"  — Pre-computed SignCLIP .npy files (best results,
#                            requires external setup)
# ============================================================================

EMBEDDING_MODE = "text_embedding"   # <-- Change this to switch modes
SIMILARITY_WEIGHT = 10.0 if EMBEDDING_MODE != "none" else 0.0

print(f"Embedding mode: {EMBEDDING_MODE}")
print(f"Similarity weight: {SIMILARITY_WEIGHT}")

In [ ]:
# ── Stage 2: Compute embeddings and similarity matrix ───────────────────
import numpy as np
from align_similarity import compute_similarity_matrix
from utils import get_subtitle_cues

# Load the subtitle cues (needed for similarity computation)
header_lines, all_cues = get_subtitle_cues(str(OUTPUT_DIR / "subtitles" / "04.vtt"))

SIM_MATRIX = None

if EMBEDDING_MODE == "none":
    print("Stage 2 skipped — no embeddings.")
    # For the auto-segmented pipeline, we use auto_signs
    SIGNS_FOR_ALIGNMENT = auto_signs

elif EMBEDDING_MODE == "text_embedding":
    # SentenceTransformer embeds both subtitle text and sign segment text.
    # This REQUIRES sign segments with text labels → use human GT signs.
    print("Using SentenceTransformer (text_embedding mode)...")
    print("Note: text_embedding requires sign segments with text labels.")
    print("      Using human Gloss Labeling signs (not auto-segmented).")
    print()

    # SentenceTransformer automatically uses GPU if available
    SIM_MATRIX = compute_similarity_matrix(
        all_cues, gt_sign_segments,
        similarity_measure="text_embedding",
    )
    # Override: use GT signs since they have text labels
    SIGNS_FOR_ALIGNMENT = gt_sign_segments

    print(f"\nSimilarity matrix shape: {SIM_MATRIX.shape}")
    print(f"  Subtitle cues (rows): {SIM_MATRIX.shape[0]}")
    print(f"  Sign segments (cols): {SIM_MATRIX.shape[1]}")
    print(f"  Value range: [{SIM_MATRIX.min():.3f}, {SIM_MATRIX.max():.3f}]")
    print(f"  Non-zero entries: {np.count_nonzero(SIM_MATRIX)} / {SIM_MATRIX.size}")

elif EMBEDDING_MODE == "sign_clip_embedding":
    # SignCLIP uses pre-computed .npy embedding files.
    # These must be generated externally (see markdown above).
    print("Using SignCLIP (sign_clip_embedding mode)...")

    sub_emb_dir = OUTPUT_DIR / "subtitle_embedding"
    seg_emb_dir = OUTPUT_DIR / "segmentation_embedding"
    sub_emb_file = sub_emb_dir / "04.npy"
    seg_emb_file = seg_emb_dir / "04.npy"

    if sub_emb_file.exists() and seg_emb_file.exists():
        subtitle_embedding = np.load(sub_emb_file)
        segmentation_embedding = np.load(seg_emb_file)
        print(f"Loaded subtitle embeddings:      {subtitle_embedding.shape}")
        print(f"Loaded segmentation embeddings:  {segmentation_embedding.shape}")

        SIM_MATRIX = compute_similarity_matrix(
            all_cues, auto_signs,
            similarity_measure="sign_clip_embedding",
            subtitle_embedding=subtitle_embedding,
            segmentation_embedding=segmentation_embedding,
        )
        SIGNS_FOR_ALIGNMENT = auto_signs

        print(f"\nSimilarity matrix shape: {SIM_MATRIX.shape}")
        print(f"  Value range: [{SIM_MATRIX.min():.3f}, {SIM_MATRIX.max():.3f}]")
    else:
        print(f"\nSignCLIP embedding files not found:")
        print(f"  Expected: {sub_emb_file}")
        print(f"  Expected: {seg_emb_file}")
        print(f"\nFalling back to 'none' mode (temporal alignment only).")
        print(f"See the markdown cell above for SignCLIP setup instructions.")
        EMBEDDING_MODE = "none"
        SIMILARITY_WEIGHT = 0.0
        SIM_MATRIX = None
        SIGNS_FOR_ALIGNMENT = auto_signs

else:
    raise ValueError(f"Unknown EMBEDDING_MODE: {EMBEDDING_MODE}")

print(f"\nSigns used for alignment: {len(SIGNS_FOR_ALIGNMENT)} segments")
print(f"Similarity weight: {SIMILARITY_WEIGHT}")

---
## Section 6 — Stage 3: DP Alignment

**Input:** `.eaf` with SIGN tier (or GT signs if using text_embedding) + `.vtt` subtitles + similarity matrix  
**Output:** aligned `.vtt` + updated `.eaf` with `SUBTITLE_SHIFTED` tier

### The DP cost function

```
Cost(i, k→j) = |cue.start − sign[k].start|          (start-time mismatch)
             + |cue.end   − sign[j].end  |            (end-time mismatch)
             + α × |cue.duration − group.duration|    (duration penalty)
             + β × Σ gaps_between_signs(k..j)         (gap penalty)
             − γ × semantic_similarity(cue_i, signs)  (similarity reward)
```

| Parameter | Value | Role |
|----|----|----|  
| `duration_penalty_weight` | 1.0 | Penalises cue/group duration mismatch |
| `gap_penalty_weight` | 5.0 | Penalises gaps between signs within a group |
| `window_size` | 50 | Number of nearest signs to search per cue |
| `max_gap` | 10.0 | Max allowed gap (seconds) in post-processing |
| `similarity_weight` | 0.0 or 10.0 | Semantic similarity reward (0 = temporal only) |

### Numba JIT

The inner DP loop uses `@njit` (Numba). First call compiles to native code (10–45s). Subsequent calls are instant.

In [ ]:
# ── Numba JIT warmup ───────────────────────────────────────────────────────
import time
from align_dp import dp_align_subtitles_to_signs

print("Compiling Numba JIT functions (first run only, 10–45s)...")
t0 = time.time()

dummy_cues = [{"start": float(i), "end": float(i+1), "mid": float(i)+0.5, "text": "x"}
              for i in range(4)]
dummy_signs = [{"start": float(i)*0.5, "end": float(i)*0.5+0.4, "mid": float(i)*0.5+0.2, "text": "y"}
               for i in range(8)]
dp_align_subtitles_to_signs(
    dummy_cues, dummy_signs, gt_cues=[],
    duration_penalty_weight=1.0, gap_penalty_weight=1.0,
    window_size=5, max_gap=2.0, similarity_weight=0, sim_matrix=None
)
print(f"Done in {time.time() - t0:.1f}s")

In [ ]:
# ── Run DP alignment ───────────────────────────────────────────────────────
from utils import get_sign_segments_from_eaf

# Reload cues (get_subtitle_cues was already called in Stage 2)
_, cues = get_subtitle_cues(str(OUTPUT_DIR / "subtitles" / "04.vtt"))
cues_original = copy.deepcopy(cues)

print(f"Input:  {len(cues)} subtitle cues")
print(f"Signs:  {len(SIGNS_FOR_ALIGNMENT)} sign segments ({EMBEDDING_MODE} mode)")
print(f"Similarity weight: {SIMILARITY_WEIGHT}")
print()
print("Running DP alignment...")
t0 = time.time()

dp_align_subtitles_to_signs(
    cues, SIGNS_FOR_ALIGNMENT,
    gt_cues=[],
    duration_penalty_weight=1.0,
    gap_penalty_weight=5.0,
    window_size=50,
    max_gap=10.0,
    similarity_weight=SIMILARITY_WEIGHT,
    sim_matrix=SIM_MATRIX,
)

print(f"Done in {time.time() - t0:.2f}s")
print(f"Aligned {len(cues)} cues.")

# Show before/after for first 8 cues
import pandas as pd
rows = []
for i in range(min(8, len(cues))):
    rows.append({
        "Cue": i,
        "Original start": f"{cues_original[i]['start']:.2f}s",
        "Aligned start": f"{cues[i]['start']:.2f}s",
        "\u0394 start": f"{cues[i]['start'] - cues_original[i]['start']:+.2f}s",
        "Text": cues[i]["text"][:40],
    })
pd.DataFrame(rows)

In [ ]:
# ── Write output files ──────────────────────────────────────────────────────
from utils import reconstruct_vtt, write_updated_eaf

# Write aligned VTT
aligned_vtt = reconstruct_vtt(header_lines, cues)
aligned_path = OUTPUT_DIR / "aligned" / "04.vtt"
with open(aligned_path, "w", encoding="utf-8") as f:
    f.write(aligned_vtt)
print(f"Aligned VTT:     {aligned_path}")

# Write updated EAF (appends SUBTITLE_SHIFTED tier to segmentation EAF)
if SEG_OUTPUT.exists():
    write_updated_eaf(str(SEG_OUTPUT), cues, "04", signs=SIGNS_FOR_ALIGNMENT)
    updated_eaf = str(SEG_OUTPUT).replace(".eaf", "_updated.eaf")
    print(f"Updated EAF:     {updated_eaf}")
print(f"\nOpen the updated .eaf in ELAN to visually inspect the alignment.")

---
## Section 7 — Evaluation

Metrics:
- **Frame-level accuracy** — % frames where subtitle presence/absence matches GT
- **F1@IoU** — F1 at intersection-over-union thresholds 0.10, 0.25, 0.50
- **Mean/median start/end offsets** — temporal error in seconds

We compare:
1. **Current pipeline** — whatever embedding mode was selected above
2. **No-embedding baseline** — auto-segmented signs with `similarity_weight=0`

In [ ]:
from evaluate_sub_alignment import eval_subtitle_alignment
from utils import print_results

# ── Evaluate: current pipeline output ──────────────────────────────────
eval_main = eval_subtitle_alignment(
    Path(str(OUTPUT_DIR / "aligned")),
    Path(str(OUTPUT_DIR / "ground_truth")),
    ["04"], VIDEO_FPS, 0, 0
)
print("=" * 60)
print(f"Pipeline result (embedding={EMBEDDING_MODE}):")
print("=" * 60)
print_results(eval_main)

In [ ]:
# ── Baseline: auto-segmented, no embeddings ────────────────────────────
cues_baseline = copy.deepcopy(cues_original)
dp_align_subtitles_to_signs(
    cues_baseline, auto_signs,
    gt_cues=[],
    duration_penalty_weight=1.0,
    gap_penalty_weight=5.0,
    window_size=50,
    max_gap=10.0,
    similarity_weight=0,
    sim_matrix=None,
)
baseline_dir = OUTPUT_DIR / "aligned_baseline"
baseline_dir.mkdir(exist_ok=True)
baseline_vtt = reconstruct_vtt(header_lines, cues_baseline)
with open(baseline_dir / "04.vtt", "w", encoding="utf-8") as f:
    f.write(baseline_vtt)

eval_baseline = eval_subtitle_alignment(
    Path(str(baseline_dir)),
    Path(str(OUTPUT_DIR / "ground_truth")),
    ["04"], VIDEO_FPS, 0, 0
)

# ── Also: alignment with human GT signs (upper bound) ───────────────────
cues_gt_signs = copy.deepcopy(cues_original)
dp_align_subtitles_to_signs(
    cues_gt_signs, gt_sign_segments,
    gt_cues=[],
    duration_penalty_weight=1.0,
    gap_penalty_weight=5.0,
    window_size=50,
    max_gap=10.0,
    similarity_weight=0,
    sim_matrix=None,
)
gt_aligned_dir = OUTPUT_DIR / "aligned_gt_signs"
gt_aligned_vtt = reconstruct_vtt(header_lines, cues_gt_signs)
with open(gt_aligned_dir / "04.vtt", "w", encoding="utf-8") as f:
    f.write(gt_aligned_vtt)

eval_gt = eval_subtitle_alignment(
    Path(str(gt_aligned_dir)),
    Path(str(OUTPUT_DIR / "ground_truth")),
    ["04"], VIDEO_FPS, 0, 0
)

# ── Side-by-side comparison ────────────────────────────────────────────
col_names = [f"Pipeline ({EMBEDDING_MODE})", "Baseline (no emb)", "Human Gloss (upper bound)"]
print("\n" + "=" * 60)
print("Three-way Comparison")
print("=" * 60)
print_results(
    [eval_main, eval_baseline, eval_gt],
    column_names=col_names
)

In [ ]:
# ── Evaluation metrics visualisation ────────────────────────────────────
import matplotlib.pyplot as plt

def parse_eval_metrics(eval_str):
    metrics = {}
    for line in eval_str.strip().split("\n"):
        if "|" not in line or "---" in line or "Metric" in line:
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 3:
            key, val = parts[1], parts[2]
            if "Frame-level" in key:
                metrics["Frame Acc."] = float(val)
            elif "F1@0.10" in key:
                metrics["F1@0.10"] = float(val)
            elif "F1@0.25" in key:
                metrics["F1@0.25"] = float(val)
            elif "F1@0.50" in key:
                metrics["F1@0.50"] = float(val)
            elif "start offset (abs)" in key:
                vals = val.split("/")
                metrics["Mean |\u0394start|"] = float(vals[0])
                metrics["Median |\u0394start|"] = float(vals[1])
            elif "end offset (abs)" in key:
                vals = val.split("/")
                metrics["Mean |\u0394end|"] = float(vals[0])
                metrics["Median |\u0394end|"] = float(vals[1])
    return metrics

m_main = parse_eval_metrics(eval_main)
m_base = parse_eval_metrics(eval_baseline)
m_gt = parse_eval_metrics(eval_gt)

if m_main and m_base and m_gt:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f"SEA Full Pipeline \u2014 Evaluation (embedding={EMBEDDING_MODE})",
                 fontsize=16, fontweight="bold")

    # Panel A: Accuracy & F1
    ax = axes[0]
    labels = ["Frame Acc.", "F1@0.10", "F1@0.25", "F1@0.50"]
    main_v = [m_main.get(k, 0) for k in labels]
    base_v = [m_base.get(k, 0) for k in labels]
    gt_v = [m_gt.get(k, 0) for k in labels]
    x = np.arange(len(labels))
    w = 0.25
    ax.bar(x - w, main_v, w, label=f"Pipeline ({EMBEDDING_MODE})", color="steelblue")
    ax.bar(x,     base_v, w, label="Baseline (no emb)", color="coral")
    ax.bar(x + w, gt_v,   w, label="Human Gloss", color="seagreen")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("%"); ax.set_ylim(0, 105); ax.legend(fontsize=9)
    ax.set_title("(A) Accuracy & F1 Scores")
    for i, (a, b, g) in enumerate(zip(main_v, base_v, gt_v)):
        ax.text(i-w, a+1, f"{a:.1f}", ha="center", va="bottom", fontsize=8)
        ax.text(i,   b+1, f"{b:.1f}", ha="center", va="bottom", fontsize=8)
        ax.text(i+w, g+1, f"{g:.1f}", ha="center", va="bottom", fontsize=8)

    # Panel B: Timing offsets
    ax = axes[1]
    labels_b = ["Mean |\u0394start|", "Median |\u0394start|", "Mean |\u0394end|", "Median |\u0394end|"]
    main_b = [m_main.get(k, 0) for k in labels_b]
    base_b = [m_base.get(k, 0) for k in labels_b]
    gt_b = [m_gt.get(k, 0) for k in labels_b]
    x = np.arange(len(labels_b))
    ax.bar(x - w, main_b, w, label=f"Pipeline ({EMBEDDING_MODE})", color="steelblue")
    ax.bar(x,     base_b, w, label="Baseline (no emb)", color="coral")
    ax.bar(x + w, gt_b,   w, label="Human Gloss", color="seagreen")
    ax.set_xticks(x); ax.set_xticklabels(labels_b, fontsize=9)
    ax.set_ylabel("Seconds"); ax.legend(fontsize=9)
    ax.set_title("(B) Absolute Timing Offsets (lower = better)")

    plt.tight_layout()
    fig.savefig(ASSETS_DIR / "evaluation_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to: {ASSETS_DIR / 'evaluation_metrics.png'}")
else:
    print("Could not parse evaluation metrics for visualisation.")

---
## Section 8 — Timeline Visualisation

Four-track chart showing a 60-second window:

1. **Auto SIGN** (steel blue) — auto-segmented sign boundaries (Stage 1)
2. **Human Gloss Labeling** (coral) — human-annotated signs (ground truth)
3. **Original CC_Aligned** (tomato) — input subtitles (speech-timed)
4. **DP-Aligned** (sea green) — output subtitles (sign-timed)

In [ ]:
T_START, T_END = 33.0, 93.0

fig, axes = plt.subplots(4, 1, figsize=(22, 10), sharex=True)
fig.suptitle(f"SEA Full Pipeline \u2014 Timeline [{T_START:.0f}s \u2013 {T_END:.0f}s]",
             fontsize=16, fontweight="bold")

tracks = [
    ("Auto SIGN (Stage 1)",       auto_signs,        "steelblue"),
    ("Human Gloss Labeling (GT)", gt_sign_segments,  "coral"),
    ("Original CC_Aligned",       cues_original,     "tomato"),
    ("DP-Aligned Output",         cues,              "seagreen"),
]

for ax, (label, segments, color) in zip(axes, tracks):
    for seg in segments:
        s, e = seg["start"], seg["end"]
        if e < T_START or s > T_END:
            continue
        s = max(s, T_START)
        e = min(e, T_END)
        ax.barh(0, e - s, left=s, height=0.6, color=color, edgecolor="white", linewidth=0.5)
        if "text" in seg and seg["text"] and (e - s) > 1.0:
            ax.text((s + e) / 2, 0, seg["text"][:15], ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold", clip_on=True)
    ax.set_ylabel(label, fontsize=10, rotation=0, ha="right", va="center")
    ax.set_yticks([])
    ax.set_xlim(T_START, T_END)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

axes[-1].set_xlabel("Time (seconds)", fontsize=12)
axes[-1].set_xticks(np.arange(T_START, T_END + 1, 5))

plt.tight_layout()
fig.savefig(ASSETS_DIR / "alignment_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to: {ASSETS_DIR / 'alignment_visualization.png'}")

---
## Section 9 — Summary and Next Steps

### What we accomplished

| Stage | Input | Output | Tool | GPU? |
|-------|-------|--------|------|------|
| **0 — Pose** | `04.mp4` | `04.pose` | `videos_to_poses` (MediaPipe) | Yes |
| **1 — Segment** | `04.pose` | `04.eaf` (SIGN tier) | `pose_to_segments` | No |
| **2 — Embed** | cues + signs | similarity matrix | SentenceTransformer / SignCLIP | Yes |
| **3 — Align** | `.eaf` + `.vtt` + sim | aligned `.vtt` + `_updated.eaf` | `dp_align_subtitles_to_signs()` | No |

### Embedding modes summary

| Mode | Works out of the box? | Needs sign text? | Expected improvement |
|------|-----------------------|-------------------|---------------------|
| `none` | Yes | No | Baseline |
| `text_embedding` | Yes | Yes (uses GT signs) | Moderate |
| `sign_clip_embedding` | No (external setup) | No (uses pose features) | **+6% F1@0.50** |

### Generated files

```
data/example_alignment/pipeline_output/
├── poses/04.pose                              ← Stage 0
├── subtitles/04.vtt                           ← Input VTT
├── ground_truth/04.vtt                        ← Ground truth
├── segmentation/E4s-1_30_70/04.eaf            ← Stage 1
├── segmentation/E4s-1_30_70/04_updated.eaf    ← Stage 3
└── aligned/04.vtt                             ← Stage 3
```

### Next steps

1. **Try different segmentation thresholds** — `SIGN_B=60, SIGN_O=50` (BOBSL defaults)
2. **Switch embedding modes** — change `EMBEDDING_MODE` in Section 5 and re-run
3. **Set up SignCLIP** — follow the instructions in Section 5 for best results
4. **Use on your own videos** — provide `.mp4` + `.vtt` and run the same pipeline
5. **Batch processing** — use `SEA/align.py --mode=inference` for multiple videos
6. **Hyperparameter tuning** — use `--mode=training` for random search